In [1]:

# ============================================================
# JEEVAN LINK - URGENCY PREDICTION MODEL TRAINING
# Algorithm: Random Forest Classifier
# ============================================================

import pandas as pd
import numpy as np
import json
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# ============================================================
# 1. FILE PATHS
# ============================================================

PROJECT_DIR = Path(r"C:\Users\ishit\jeevan_link_ml")

DATA_FILE = (
    PROJECT_DIR
    / "Data"
    / "Processed_data"
    / "sos_urgency_dataset_model"
    / "sos_urgency_dataset_final_cleaned.csv"
)

MODEL_DIR = PROJECT_DIR / "Models" / "Urgency_Model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE}\n"
        "Check the exact dataset filename and location."
    )

# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

# ============================================================
# 3. DEFINE TARGET AND FEATURES
# ============================================================

TARGET = "urgency_label"

if TARGET not in df.columns:
    raise ValueError("urgency_label column not found.")

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str).str.strip()

# Ensure all input features are numeric
X = X.apply(pd.to_numeric, errors="raise")

# Check missing values
if X.isnull().any().any() or y.isnull().any():
    raise ValueError("Missing values found. Review the dataset first.")

print("\nInput features:", X.shape[1])
print("Target classes:", sorted(y.unique()))

# ============================================================
# 4. CHECK TARGET DISTRIBUTION
# ============================================================

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

# ============================================================
# 5. SPLIT DATASET
# 70% TRAINING
# 15% VALIDATION
# 15% TESTING
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nDATA SPLIT")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

# ============================================================
# 6. BUILD RANDOM FOREST MODEL
# ============================================================

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

# ============================================================
# 7. TRAIN MODEL
# ============================================================

print("\nTraining Urgency Prediction Model...")

model.fit(X_train, y_train)

print("Model training completed!")

# ============================================================
# 8. VALIDATION EVALUATION
# ============================================================

val_predictions = model.predict(X_val)

val_accuracy = accuracy_score(y_val, val_predictions)
val_macro_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro",
    zero_division=0
)

print("\nVALIDATION RESULTS")
print("==============================")
print("Accuracy:", round(val_accuracy, 4))
print("Macro F1:", round(val_macro_f1, 4))

print("\nValidation Classification Report:")
print(
    classification_report(
        y_val,
        val_predictions,
        zero_division=0
    )
)

# ============================================================
# 9. FINAL TEST EVALUATION
# ============================================================

test_predictions = model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_predictions)

test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)

print("\nFINAL TEST RESULTS")
print("==============================")
print("Accuracy:", round(test_accuracy, 4))
print("Macro F1:", round(test_macro_f1, 4))
print("Weighted F1:", round(test_weighted_f1, 4))

print("\nTest Classification Report:")

test_report = classification_report(
    y_test,
    test_predictions,
    zero_division=0
)

print(test_report)

# ============================================================
# 10. CONFUSION MATRIX
# ============================================================

class_names = ["Low", "Medium", "High", "Critical"]

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=class_names
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print("\nCONFUSION MATRIX")
print(cm_df)

cm_df.to_csv(MODEL_DIR / "urgency_confusion_matrix.csv")

# ============================================================
# 11. FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

print("\nTOP FEATURE IMPORTANCE")
print(feature_importance.to_string(index=False))

feature_importance.to_csv(
    MODEL_DIR / "urgency_feature_importance.csv",
    index=False
)

# ============================================================
# 12. SAVE TRAINED MODEL
# ============================================================

MODEL_FILE = MODEL_DIR / "urgency_model.pkl"

joblib.dump(model, MODEL_FILE)

# Save feature names for future predictions
joblib.dump(
    list(X.columns),
    MODEL_DIR / "urgency_feature_names.pkl"
)

# ============================================================
# 13. SAVE EVALUATION METRICS
# ============================================================

metrics = {
    "algorithm": "Random Forest Classifier",
    "dataset_rows": int(len(df)),
    "feature_count": int(X.shape[1]),
    "classes": class_names,
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_val)),
    "test_rows": int(len(X_test)),
    "validation_accuracy": float(val_accuracy),
    "validation_macro_f1": float(val_macro_f1),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1)
}

with open(
    MODEL_DIR / "urgency_model_metrics.json",
    "w"
) as file:
    json.dump(metrics, file, indent=4)

# ============================================================
# 14. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("JEEVAN LINK URGENCY MODEL TRAINING COMPLETED")
print("=" * 60)

print("\nModel saved at:")
print(MODEL_FILE)

print("\nAll output files:")

for file in MODEL_DIR.iterdir():
    print(file.name)

print("\nTraining, validation and testing completed.")


Dataset loaded successfully!
Dataset shape: (4154, 19)

Columns:
['people_affected', 'people_trapped', 'injuries_reported', 'medical_emergency', 'rescue_required', 'food_required', 'water_required', 'shelter_required', 'urgency_label', 'water_level_Rapidly_Rising', 'water_level_Rising', 'water_level_Stable', 'building_condition_Collapsing', 'building_condition_Damaged', 'building_condition_Intact', 'building_condition_Unknown', 'location_risk_High', 'location_risk_Low', 'location_risk_Medium']

Input features: 18
Target classes: ['Critical', 'High', 'Low', 'Medium']

Target distribution:
urgency_label
High        1989
Critical    1135
Medium       945
Low           85
Name: count, dtype: int64

Target percentages:
urgency_label
High        47.88
Critical    27.32
Medium      22.75
Low          2.05
Name: proportion, dtype: float64

DATA SPLIT
Training: (2907, 18)
Validation: (623, 18)
Testing: (624, 18)

Training target distribution:
urgency_label
High        1392
Critical     794
Medi